# Session 1: Introduction to MongoDB

## Data Ecosystems and Governance in Organizations
**MSc Business Analytics | Nova School of Business and Economics**

---

### ⏱️ Estimated Time: 30-40 minutes

| Section | Time |
|---------|------|
| Setup & Connection | 5-10 min |
| CRUD Walkthrough | 15-20 min |
| Exercises | 15-20 min |

### 📋 What You'll Do
1. Install and start MongoDB
2. Connect using PyMongo
3. Practice CRUD operations
4. Complete 4 exercises

> 💡 **Note:** The theory (NoSQL types, CAP theorem) was covered in the lecture slides. This notebook focuses on **hands-on practice**.

---
# Part 1: MongoDB Installation
---

Choose your operating system and follow the instructions below.

## 🍎 macOS Installation

### Step 1: Install Homebrew (if not installed)
```bash
/bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"
```

### Step 2: Install MongoDB
```bash
# Add MongoDB tap
brew tap mongodb/brew

# Install MongoDB Community Edition
brew install mongodb-community@7.0
```

### Step 3: Start MongoDB
```bash
# Start as a service (recommended - runs in background)
brew services start mongodb-community@7.0

# OR start manually (runs in foreground)
mongod --config /opt/homebrew/etc/mongod.conf   # Apple Silicon (M1/M2/M3)
mongod --config /usr/local/etc/mongod.conf      # Intel Mac
```

### Step 4: Verify Installation
```bash
# Open a new terminal and run:
mongosh

# You should see the MongoDB shell. Type 'exit' to quit.
```

### Stopping MongoDB (when done)
```bash
brew services stop mongodb-community@7.0
```

## 🪟 Windows Installation

### Step 1: Download MongoDB
1. Go to: https://www.mongodb.com/try/download/community
2. Select:
   - Version: 7.0 (current)
   - Platform: Windows
   - Package: MSI
3. Click **Download**

### Step 2: Install MongoDB
1. Run the downloaded `.msi` installer
2. Choose **Complete** installation
3. ✅ Check **Install MongoDB as a Service** (important!)
4. ✅ Check **Install MongoDB Compass** (optional GUI tool)
5. Complete the installation

### Step 3: Verify Installation
MongoDB runs automatically as a Windows service.

```powershell
# Open PowerShell and run:
mongosh

# You should see the MongoDB shell. Type 'exit' to quit.
```

### If `mongosh` is not recognized:
Add MongoDB to your PATH:
1. Search "Environment Variables" in Windows
2. Edit PATH, add: `C:\Program Files\MongoDB\Server\7.0\bin`
3. Restart PowerShell

### Starting/Stopping the Service
```powershell
# Check status
Get-Service MongoDB

# Start service
Start-Service MongoDB

# Stop service
Stop-Service MongoDB
```

## 🐧 Linux (Ubuntu/Debian) Installation

### Step 1: Import MongoDB GPG Key
```bash
curl -fsSL https://www.mongodb.org/static/pgp/server-7.0.asc | \
   sudo gpg -o /usr/share/keyrings/mongodb-server-7.0.gpg --dearmor
```

### Step 2: Add Repository
```bash
# For Ubuntu 22.04 (Jammy)
echo "deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg ] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" | \
   sudo tee /etc/apt/sources.list.d/mongodb-org-7.0.list
```

### Step 3: Install MongoDB
```bash
sudo apt-get update
sudo apt-get install -y mongodb-org
```

### Step 4: Start MongoDB
```bash
# Start the service
sudo systemctl start mongod

# Enable auto-start on boot
sudo systemctl enable mongod

# Check status
sudo systemctl status mongod
```

### Step 5: Verify Installation
```bash
mongosh
```

## ☁️ Alternative: MongoDB Atlas (Cloud - No Installation)

If you have trouble installing MongoDB locally, use the free cloud option:

### Step 1: Create Account
1. Go to: https://www.mongodb.com/cloud/atlas/register
2. Sign up for free

### Step 2: Create a Cluster
1. Click **Build a Database**
2. Choose **M0 FREE** tier
3. Select a cloud provider and region (closest to you)
4. Click **Create**

### Step 3: Setup Access
1. Create a database user (remember the password!)
2. Add your IP address to the whitelist (or allow access from anywhere for testing)

### Step 4: Get Connection String
1. Click **Connect** on your cluster
2. Choose **Connect your application**
3. Copy the connection string

### Step 5: Use in Python
```python
# Replace <password> with your database user password
client = MongoClient("mongodb+srv://username:<password>@cluster0.xxxxx.mongodb.net/")
```

## 🐍 Install PyMongo

Run this cell to install the Python driver:

In [8]:
# Install PyMongo (run once)

import sys
!{sys.executable} -m pip install pymongo

# !pip install pymongo


---
# Part 2: Connect to MongoDB
---

⚠️ **Make sure MongoDB is running before executing the cell below!**

In [9]:
# Import required libraries
from pymongo import MongoClient
from datetime import datetime
from pprint import pprint

# Connect to local MongoDB
# If using Atlas, replace with your connection string
#client = MongoClient("mongodb+srv://<user>:<password>@<cluster-url>/?retryWrites=true&w=majority")
client = MongoClient('localhost', 27017)

# Create/access database and collection
db = client['dego_session1']
products = db['products']

# Clear collection for fresh start
products.delete_many({})

print("✅ Connected to MongoDB!")
print(f"   Database: {db.name}")
print(f"   Collection: products")

✅ Connected to MongoDB!
   Database: dego_session1
   Collection: products


### ❌ Connection Failed?

If you see an error like `ServerSelectionTimeoutError`, try these fixes:

1. **Check if MongoDB is running:**
   - macOS: `brew services list` (look for mongodb-community)
   - Windows: `Get-Service MongoDB` in PowerShell
   - Linux: `sudo systemctl status mongod`

2. **Start MongoDB if it's not running:**
   - macOS: `brew services start mongodb-community@7.0`
   - Windows: `Start-Service MongoDB`
   - Linux: `sudo systemctl start mongod`

3. **Still not working?** Use MongoDB Atlas (cloud) as backup.

---
# Part 3: CRUD Operations
---

**CRUD** = **C**reate, **R**ead, **U**pdate, **D**elete

| Operation | Methods |
|-----------|--------|
| **C**reate | `insert_one()`, `insert_many()` |
| **R**ead | `find()`, `find_one()` |
| **U**pdate | `update_one()`, `update_many()` |
| **D**elete | `delete_one()`, `delete_many()` |

## Create: Inserting Documents

In [10]:
# Insert a single document
result = products.insert_one({
    "name": "Wireless Mouse",
    "category": "Electronics",
    "price": 29.99,
    "stock": 150
})

print(f"Inserted document with _id: {result.inserted_id}")

Inserted document with _id: 698b1f8c8aa0547507a3dfc8


In [11]:
# Insert multiple documents
products_to_insert = [
    {"name": "Mechanical Keyboard", "category": "Electronics", "price": 89.99, "stock": 75},
    {"name": "USB-C Hub", "category": "Electronics", "price": 45.00, "stock": 200},
    {"name": "Office Chair", "category": "Furniture", "price": 299.99, "stock": 30},
    {"name": "Standing Desk", "category": "Furniture", "price": 549.00, "stock": 15},
    {"name": "Notebook Set", "category": "Office Supplies", "price": 12.99, "stock": 500}
]

result = products.insert_many(products_to_insert)
print(f"Inserted {len(result.inserted_ids)} documents")

Inserted 5 documents


## Read: Querying Documents

In [12]:
# Find all documents
print("All products:")
print("-" * 60)
for product in products.find():
    print(f"{product['name']:25} | ${product['price']:>7.2f} | Stock: {product['stock']}")

All products:
------------------------------------------------------------
Wireless Mouse            | $  29.99 | Stock: 150
Mechanical Keyboard       | $  89.99 | Stock: 75
USB-C Hub                 | $  45.00 | Stock: 200
Office Chair              | $ 299.99 | Stock: 30
Standing Desk             | $ 549.00 | Stock: 15
Notebook Set              | $  12.99 | Stock: 500


In [13]:
# Find with filter
print("Electronics only:")
for product in products.find({"category": "Electronics"}):
    print(f"  - {product['name']}")

Electronics only:
  - Wireless Mouse
  - Mechanical Keyboard
  - USB-C Hub


In [14]:
# Find with comparison operator
print("Products over $50:")
for product in products.find({"price": {"$gt": 50}}):
    print(f"  - {product['name']}: ${product['price']}")

Products over $50:
  - Mechanical Keyboard: $89.99
  - Office Chair: $299.99
  - Standing Desk: $549.0


In [15]:
# Projection: select specific fields only
print("Names and prices only (excluding _id):")
for product in products.find({}, {"name": 1, "price": 1, "_id": 0}):
    print(f"  {product}")

Names and prices only (excluding _id):
  {'name': 'Wireless Mouse', 'price': 29.99}
  {'name': 'Mechanical Keyboard', 'price': 89.99}
  {'name': 'USB-C Hub', 'price': 45.0}
  {'name': 'Office Chair', 'price': 299.99}
  {'name': 'Standing Desk', 'price': 549.0}
  {'name': 'Notebook Set', 'price': 12.99}


In [16]:
# Find one document
print("Finding one product:")
product = products.find_one({"name": "Wireless Mouse"})
pprint(product)

Finding one product:
{'_id': ObjectId('698b1f8c8aa0547507a3dfc8'),
 'category': 'Electronics',
 'name': 'Wireless Mouse',
 'price': 29.99,
 'stock': 150}


### Common Query Operators

| Operator | Description | Example |
|----------|-------------|--------|
| `$gt` | Greater than | `{"price": {"$gt": 100}}` |
| `$gte` | Greater than or equal | `{"price": {"$gte": 100}}` |
| `$lt` | Less than | `{"stock": {"$lt": 50}}` |
| `$lte` | Less than or equal | `{"stock": {"$lte": 50}}` |
| `$ne` | Not equal | `{"status": {"$ne": "cancelled"}}` |
| `$in` | In array | `{"category": {"$in": ["A", "B"]}}` |

## Update: Modifying Documents

In [17]:
# Update single document with $set
result = products.update_one(
    {"name": "Wireless Mouse"},      # filter
    {"$set": {"price": 24.99}}       # update operation
)

print(f"Matched: {result.matched_count}, Modified: {result.modified_count}")

# Verify
product = products.find_one({"name": "Wireless Mouse"})
print(f"New price: ${product['price']}")

Matched: 1, Modified: 1
New price: $24.99


In [18]:
# Increment a value with $inc
result = products.update_one(
    {"name": "Wireless Mouse"},
    {"$inc": {"stock": 50}}          # Add 50 to stock
)

product = products.find_one({"name": "Wireless Mouse"})
print(f"New stock: {product['stock']}")

New stock: 200


In [19]:
# Update multiple documents
result = products.update_many(
    {"category": "Electronics"},
    {"$set": {"on_sale": True}}
)

print(f"Updated {result.modified_count} documents with 'on_sale' flag")

# Verify
for product in products.find({"category": "Electronics"}):
    print(f"  - {product['name']}: on_sale={product.get('on_sale', False)}")

Updated 3 documents with 'on_sale' flag
  - Wireless Mouse: on_sale=True
  - Mechanical Keyboard: on_sale=True
  - USB-C Hub: on_sale=True


### Common Update Operators

| Operator | Description | Example |
|----------|-------------|--------|
| `$set` | Set field value | `{"$set": {"price": 99}}` |
| `$inc` | Increment by amount | `{"$inc": {"stock": -1}}` |
| `$push` | Add to array | `{"$push": {"tags": "new"}}` |
| `$pull` | Remove from array | `{"$pull": {"tags": "old"}}` |
| `$unset` | Remove field | `{"$unset": {"temp_field": ""}}` |

## Delete: Removing Documents

In [ ]:
# First, let's add a product with zero stock
products.insert_one({
    "name": "Discontinued Item",
    "category": "Electronics",
    "price": 9.99,
    "stock": 0
})

print("Products with zero stock:")
for product in products.find({"stock": 0}):
    print(f"  - {product['name']}")

In [ ]:
# Delete products with zero stock
result = products.delete_many({"stock": 0})
print(f"Deleted {result.deleted_count} document(s)")
print(f"Remaining products: {products.count_documents({})}")

> 💡 **Practical Tip:** Consider using **soft deletes** when you might need the data later:
> ```python
> # Instead of delete_one():
> update_one({"_id": id}, {"$set": {"deleted": True, "deleted_at": datetime.now()}})
> ```

---
# Part 4: Exercises
---

Complete the exercises below. Each exercise builds on your CRUD knowledge.

## Exercise 1: Insert Documents

**Task:** Insert 3 new products in the "Office Supplies" category.

Each product should have: `name`, `category`, `price`, `stock`

Use `insert_many()` to insert all 3 at once.

In [ ]:
# YOUR CODE HERE



In [ ]:
# Verify: Count Office Supplies
count = products.count_documents({"category": "Office Supplies"})
print(f"Office Supplies count: {count}")  # Should be 4 (1 existing + 3 new)

## Exercise 2: Query with Filters

**Task:** Find all products priced between $25 and $100 (inclusive) with stock > 50.

*Hint: Use `$gte`, `$lte`, and `$gt` operators*

In [ ]:
# YOUR CODE HERE



## Exercise 3: Update Documents

**Task:** Add a `last_updated` field with the current timestamp to all Furniture items.

*Hint: Use `update_many()` with `$set` and `datetime.now()`*

In [ ]:
# YOUR CODE HERE



In [ ]:
# Verify: Check a Furniture item
furniture = products.find_one({"category": "Furniture"})
print(f"Furniture item: {furniture['name']}")
print(f"Last updated: {furniture.get('last_updated', 'NOT SET')}")

## Exercise 4: Calculate Inventory Value

**Task:** Calculate the total inventory value (price × stock) by category.

Loop through all products and accumulate values by category.

*Note: We'll learn a more efficient way (aggregation pipelines) in Session 2!*

In [ ]:
# YOUR CODE HERE



---
# Summary
---

## What You Learned

✅ **MongoDB Setup** — Install, start, and connect  
✅ **Create** — `insert_one()`, `insert_many()`  
✅ **Read** — `find()`, `find_one()`, query operators  
✅ **Update** — `update_one()`, `update_many()`, `$set`, `$inc`  
✅ **Delete** — `delete_one()`, `delete_many()`  

## Quick Reference

```python
# Connect
client = MongoClient('localhost', 27017)
db = client['database_name']
collection = db['collection_name']

# CRUD
collection.insert_one({...})
collection.find({"field": "value"})
collection.update_one({filter}, {"$set": {...}})
collection.delete_one({filter})
```

## Next Session Preview

- Advanced query operators (`$and`, `$or`, `$in`, `$elemMatch`)
- **Aggregation pipelines** — powerful data transformation
- Data modeling (embedding vs. referencing)
- Indexing for performance

---
## 🧹 Cleanup (Optional)

Run this cell to delete the test database when you're done:

In [ ]:
# Uncomment to delete the database
# client.drop_database('dego_session1')
# print("Database deleted!")